# 1. Data extraction

In [ ]:
import pandas as pd
from accelerate.test_utils.scripts.test_script import training_check
from torch.utils.tensorboard import SummaryWriter

from config import ROOT_DIR

In [ ]:
dir_path = ROOT_DIR.joinpath('data/Kaggle/Digit Recognizer/prepared')
train = pd.read_csv(dir_path / 'train.csv')
train.head()

# 2. Data analysis

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torchvision.transforms as T
import torch

In [ ]:
train_a = train.to_numpy()
xda = train_a[:, 1:].reshape(-1, 28, 28)
yda = train_a[:, 0]
xda.shape, yda.shape

In [ ]:
plt.imshow(xda[3], cmap='gray')

In [ ]:
xda.min(), xda.max()

## Augmentation check

In [ ]:
sample_arr = np.zeros(shape=(train['label'].nunique(), 28, 28))

for i in range(len(sample_arr)):
    condition = train['label'] == i
    sample = train[condition].sample(1, random_state=42).to_numpy()[0, 1:].reshape(28, 28)
    sample_arr[i] = sample

def draw_sample(data):
    fig, axes = plt.subplots(2, 5, figsize=(10, 4))
    axes_flat = axes.flatten()
    for idx, ax in enumerate(axes_flat):
        ax.imshow(data[idx], cmap='gray')

draw_sample(sample_arr)

In [ ]:
aug = T.RandomRotation(30)
sample_aug = aug(torch.from_numpy(sample_arr))
draw_sample(sample_aug)

In [ ]:
aug = T.RandomHorizontalFlip(p=1)
sample_aug = aug(torch.from_numpy(sample_arr))
draw_sample(sample_aug)

In [ ]:
aug = T.Lambda(lambda x: x + 0.05 * torch.randn_like(x))
temp = sample_arr / sample_arr.max()
sample_aug = aug(torch.from_numpy(temp))
draw_sample(sample_aug)

In [ ]:
aug = T.RandomAffine(degrees=0, translate=(0.1, 0.1))
sample_aug = aug(torch.from_numpy(sample_arr))
draw_sample(sample_aug)

In [ ]:
aug = T.RandomAffine(degrees=0, scale=(0.8, 1.2))
sample_aug = aug(torch.from_numpy(sample_arr))
draw_sample(sample_aug)

Pixels should be scaled.

Pixels can be near the boarder. Model should use padding.

Flip can damage 2 and 5.

# 3. Data preparation

In [ ]:
import torch
import torchvision.transforms as T
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
raw_train = train.copy()
raw_train, raw_valid = train_test_split(raw_train, test_size=0.2, shuffle=True)

y_train = torch.from_numpy(raw_train['label'].to_numpy())
y_valid = torch.from_numpy(raw_valid['label'].to_numpy())

raw_train = raw_train.drop(columns=['label']).to_numpy()
raw_valid = raw_valid.drop(columns=['label']).to_numpy()
raw_train.shape, raw_valid.shape, y_train.shape, y_valid.shape

In [ ]:
TRAIN_MAX_VALUE = raw_train.max()
train_transform = T.Compose([
    T.Lambda(lambda x: x / TRAIN_MAX_VALUE),
    T.RandomRotation(30),
    T.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    T.RandomAffine(degrees=0, scale=(0.8, 1.2)),
    T.Lambda(lambda x: x + 0.04 * torch.randn_like(x)),
])
valid_transform = T.Compose([
    T.Lambda(lambda x: x / TRAIN_MAX_VALUE),
])

class DRDataset(torch.utils.data.Dataset):
    def __init__(self, x: np.array, y: torch.tensor, transform: T.Compose):
        self.x = x
        self.y = y
        self.transform = transform

    def __len__(self):
        return self.y.shape[0]

    def __getitem__(self, index):
        x = torch.from_numpy(self.x[index].reshape(1, 28, 28)).float()
        if self.transform:
            x = self.transform(x)
        return x, self.y[index]


train_ds = DRDataset(raw_train, y_train, train_transform)
valid_ds = DRDataset(raw_valid, y_valid, valid_transform)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
valid_dl = DataLoader(valid_ds, batch_size=64, shuffle=False)
# temp, _ = train_ds[0]
# plt.imshow(temp.permute(1,2,0), cmap='gray')
# temp.max()

# 4. Model training and evaluation

In [ ]:
from torchinfo import summary
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from sklearn.metrics import accuracy_score

In [ ]:
class SimpleCNN(torch.nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()

        self.conv1 = torch.nn.Conv2d(1, 16, 3, padding=1)
        self.relu1 = torch.nn.ReLU()
        self.max_pool1 = torch.nn.MaxPool2d(2)

        self.conv2 = torch.nn.Conv2d(16, 32, 3, padding=1)
        self.relu2 = torch.nn.ReLU()
        self.max_pool2 = torch.nn.MaxPool2d(2)

        self.conv3 = torch.nn.Conv2d(32, 32, 3, padding=1)
        self.relu3 = torch.nn.ReLU()
        self.max_pool3 = torch.nn.MaxPool2d(2)

        self.conv4 = torch.nn.Conv2d(32, 16, 3, padding=1)
        self.relu4 = torch.nn.ReLU()
        self.max_pool4 = torch.nn.MaxPool2d(2)

        self.ada_max_pool5 = torch.nn.AdaptiveAvgPool2d(1)
        self.flatten5 = torch.nn.Flatten()
        self.linear5 = torch.nn.Linear(16, train['label'].nunique())

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.max_pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.max_pool2(x)

        x = self.conv3(x)
        x = self.relu3(x)
        x = self.max_pool3(x)

        x = self.conv4(x)
        x = self.relu4(x)
        x = self.max_pool4(x)

        x = self.ada_max_pool5(x)
        x = self.flatten5(x)
        x = self.linear5(x)
        return x

summary(SimpleCNN())

In [ ]:
class ModelTraining:
    def __init__(self):
        self.model = SimpleCNN().to('cuda')
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=0.001, weight_decay=0.0005)
        self.loss_fn = torch.nn.CrossEntropyLoss(reduction='none')

        self.train_dl = train_dl
        self.valid_dl = valid_dl

    def train_one_epoch(self):
        metrics = torch.zeros(3, len(self.train_dl.dataset))
        self.model.train()
        for i, (x, y) in enumerate(self.train_dl):
            x = x.to('cuda')
            y = y.to('cuda')
            self.optimizer.zero_grad()
            o = self.model(x)
            loss = self.loss_fn(o, y)
            loss.mean().backward()
            self.optimizer.step()

            batch_len = y.shape[0]
            start_idx = i * batch_len
            end_idx = start_idx + batch_len
            metrics[0, start_idx:end_idx] = o.argmax(axis=1)
            metrics[1, start_idx:end_idx] = y
            metrics[2, start_idx:end_idx] = loss
        return metrics.detach()

    def validate(self):
        with torch.no_grad():
            metrics = torch.zeros(3, len(self.valid_dl.dataset))
            self.model.eval()
            for i, (x, y) in enumerate(self.valid_dl):
                x = x.to('cuda')
                y = y.to('cuda')
                o = self.model(x)
                loss = self.loss_fn(o, y)

                batch_len = y.shape[0]
                start_idx = i * batch_len
                end_idx = start_idx + batch_len
                metrics[0, start_idx:end_idx] = o.argmax(axis=1)
                metrics[1, start_idx:end_idx] = y
                metrics[2, start_idx:end_idx] = loss
        return metrics

    def run(self, epochs=5):
        writer = SummaryWriter('logs/simple_cnn_{}'.format(datetime.now().strftime("%Y-%m-%d_%H-%M-%S")))

        for epoch in range(epochs):
            train_metrics = self.train_one_epoch()
            valid_metrics = self.validate()
            train_loss = train_metrics[2].mean()
            valid_loss = valid_metrics[2].mean()
            train_accuracy = accuracy_score(train_metrics[1], train_metrics[0])
            valid_accuracy = accuracy_score(valid_metrics[1], valid_metrics[0])
            print(epoch+1, train_loss, valid_loss, train_accuracy, valid_accuracy)

            writer.add_scalar('train/loss', train_loss, epoch + 1)
            writer.add_scalar('valid/loss', valid_loss, epoch + 1)
            writer.add_scalar('train/accuracy', train_accuracy, epoch + 1)
            writer.add_scalar('valid/accuracy', valid_accuracy, epoch + 1)
        writer.close()

model_training = ModelTraining()
model_training.run(85)

# 5. Kaggle submission

In [ ]:
dir_path = ROOT_DIR.joinpath('data/Kaggle/Digit Recognizer/prepared')
test = pd.read_csv(dir_path / 'test.csv')
test.head()

In [ ]:
raw_test = test.to_numpy()
raw_test.shape

In [ ]:
y_test = torch.zeros(raw_test.shape[0])
test_ds = DRDataset(raw_test, y_test, valid_transform)
# test_dl = DataLoader(test_ds, batch_size=64, shuffle=False)
temp, _ = test_ds[3]
plt.imshow(temp.permute(1,2,0), cmap='gray')
temp.max()

In [ ]:
results = np.zeros(raw_test.shape[0])
local_model = model_training.model
local_model.eval()
for i, (img, _) in enumerate(test_ds):
    o = local_model(img.to('cuda').unsqueeze(0)).to('cpu').detach().numpy()
    results[i] = o.argmax()

In [ ]:
dir_path = ROOT_DIR.joinpath('data/Kaggle/Digit Recognizer/prepared')
sample_submission = pd.read_csv(dir_path / 'sample_submission.csv')
sample_submission.head()

In [ ]:
sample_submission['Label'] = results.astype(int)
sample_submission

In [ ]:
dir_path = ROOT_DIR.joinpath('data/Kaggle/Digit Recognizer/results')
sample_submission.to_csv(dir_path / 'submission.csv', index=False)
print("Your submission was successfully saved!")